# Thắng


# NOTEBOOK 01: DATA UNDERSTANDING (01_data_understanding.ipynb)
**Mục tiêu:** Hiểu bài toán và bộ dữ liệu, đánh giá chất lượng dữ liệu, xác định cấu trúc dữ liệu và chuẩn bị cho các bước xử lý tiếp theo.


## I. Hiểu bài toán
- **Giới thiệu bài toán AI cần giải quyết:** Dự báo thời gian giao hàng thực tế (Delivery Time) cho các đơn hàng thương mại điện tử trên sàn Olist Brazil. Việc dự báo chính xác giúp cải thiện trải nghiệm khách hàng, lập kế hoạch logistics và kho bãi tối ưu.
- **Biến mục tiêu (Target):** `delivery_time` (Số ngày thực tế từ khi mua hàng đến khi khách hàng nhận được hàng).


## II. Tìm hiểu Dataset
- **Mô tả Dataset:** Bộ dữ liệu Olist Brazil chứa thông tin của hơn 100,000 đơn hàng từ năm 2016 đến 2018 tại Brazil, liên kết giữa khách hàng, đơn đặt hàng, thanh toán, đánh giá, chi tiết mặt hàng, sản phẩm, người bán và tọa độ địa lý.
- **Danh sách các file dữ liệu:** 9 file CSV thô.
- **Phân loại bảng dữ liệu:**
  - **Bảng chính (Fact Table):** `orders` (bảng đơn hàng) và `order_items` (chi tiết sản phẩm trong đơn).
  - **Bảng phụ (Dimension / Transaction Tables):** `customers` (khách hàng), `sellers` (người bán), `products` (sản phẩm), `geolocation` (địa lý), `order_payments` (thanh toán), `order_reviews` (đánh giá), `category_translation` (dịch tên danh mục).
  - **Mức dữ liệu (Grain):**
    - Bảng `orders`: 1 dòng = 1 đơn hàng của 1 khách hàng.
    - Bảng `order_items`: 1 dòng = 1 mặt hàng trong đơn hàng.

### 1. Danh sách toàn bộ bảng
| Bảng | Số dòng | Số cột | Vai trò |
|---|---|---|---|
| `orders` | 99,441 | 8 | Bảng chính (Thông tin đơn hàng) |
| `order_items` | 112,650 | 7 | Bảng chính (Chi tiết sản phẩm đơn hàng) |
| `customers` | 99,441 | 5 | Bảng phụ (Thông tin khách hàng) |
| `sellers` | 3,095 | 4 | Bảng phụ (Thông tin người bán) |
| `products` | 32,951 | 9 | Bảng phụ (Thông tin sản phẩm) |
| `geolocation` | 1,000,163 | 5 | Bảng phụ (Tọa độ địa lý zip code) |
| `order_payments` | 103,886 | 5 | Bảng phụ (Thông tin thanh toán) |
| `order_reviews` | 99,224 | 7 | Bảng phụ (Đánh giá của khách hàng) |
| `category_translation` | 71 | 2 | Bảng phụ (Dịch danh mục tiếng Anh) |

### 2. Sơ đồ quan hệ (ER đơn giản)
```
   customers (customer_id) ────┐
                               │
                             orders (order_id) ─── order_reviews
                               │ 
   sellers (seller_id) ────────┼─── order_items (product_id) ─── products ─── category_translation
                               │
   geolocation (zip_code) ─────┴─── order_payments
```

### 3. Xác định bảng trung tâm (Fact Table)
- Bảng `orders` và `order_items` chứa toàn bộ thông tin giao dịch cốt lõi, là các bảng trung tâm (Fact Tables) của bài toán.

### 4. Data Dictionary (mô tả các cột dữ liệu quan trọng)
- **Bảng `orders`:**
  - `order_id`: Mã duy nhất của đơn hàng.
  - `customer_id`: Mã khách hàng liên kết với đơn hàng.
  - `order_status`: Trạng thái đơn hàng (delivered, shipped, canceled, etc.).
  - `order_purchase_timestamp`: Thời điểm khách đặt mua.
  - `order_delivered_customer_date`: Thời điểm khách nhận được hàng (Target).
  - `order_estimated_delivery_date`: Thời điểm dự kiến giao hàng.
- **Bảng `order_items`:**
  - `order_id`: Mã đơn hàng.
  - `product_id`: Mã sản phẩm.
  - `seller_id`: Mã người bán.
  - `price`: Giá bán sản phẩm.
  - `freight_value`: Phí vận chuyển.

### 🔑 Bản đồ Khóa chính - Khóa ngoại (PK-FK Keys) & Các Cột Quan Trọng:

| Bảng dữ liệu | Khóa chính (PK) | Khóa ngoại (FK) | Cột quan trọng cần chú ý |
|---|---|---|---|
| `orders` (Đơn hàng) | `order_id` | `customer_id` | `order_status`, `order_purchase_timestamp`, `order_delivered_customer_date` (Target) |
| `order_items` (Chi tiết đơn) | `order_id`, `order_item_id` | `order_id`, `product_id`, `seller_id` | `price`, `freight_value` |
| `customers` (Khách hàng) | `customer_id` | `customer_zip_code_prefix` | `customer_unique_id`, `customer_state` |
| `sellers` (Người bán) | `seller_id` | `seller_zip_code_prefix` | `seller_state` |
| `products` (Sản phẩm) | `product_id` | `product_category_name` | `product_weight_g` |
| `order_payments` (Thanh toán) | Không có | `order_id` | `payment_type`, `payment_installments`, `payment_value` |
| `order_reviews` (Đánh giá) | `review_id` | `order_id` | `review_score` |



## III. Khảo sát dữ liệu của bảng chính
### 1. Đọc dữ liệu
Tiến hành nạp 9 file CSV thô vào DataFrame và hiển thị dòng dữ liệu đầu tiên.

In [1]:
# Nhập thư viện cần thiết
import pandas as pd
import os
from IPython.display import display

# Đường dẫn đến thư mục chứa dữ liệu
data_dir = '../data/archive/'

# Nạp 9 file CSV vào biến DataFrame
df_customers = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
df_geolocation = pd.read_csv(os.path.join(data_dir, 'olist_geolocation_dataset.csv'))
df_order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
df_order_payments = pd.read_csv(os.path.join(data_dir, 'olist_order_payments_dataset.csv'))
df_order_reviews = pd.read_csv(os.path.join(data_dir, 'olist_order_reviews_dataset.csv'))
df_orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
df_products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
df_sellers = pd.read_csv(os.path.join(data_dir, 'olist_sellers_dataset.csv'))
df_category_translation = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))

dataframes = {
    'customers': df_customers,
    'geolocation': df_geolocation,
    'order_items': df_order_items,
    'order_payments': df_order_payments,
    'order_reviews': df_order_reviews,
    'orders': df_orders,
    'products': df_products,
    'sellers': df_sellers,
    'category_translation': df_category_translation
}
print('Đã nạp xong', len(dataframes), 'bảng dữ liệu.')


Đã nạp xong 9 bảng dữ liệu.


**Nhận xét:**
- Cả 9 bảng dữ liệu từ bộ dataset Olist đã được nạp thành công và lưu vào dictionary `dataframes`.
- Đường dẫn dữ liệu `../data/archive/` hoạt động chính xác.
- Quy mô của các bảng rất đa dạng, phản ánh cấu trúc dữ liệu của một hệ thống thương mại điện tử lớn.


In [ ]:
print('--- Một vài dòng đầu của bảng orders ---')
display(df_orders.head(3))
print('\n--- Một vài dòng đầu của bảng order_items ---')
display(df_order_items.head(3))


### 2. Kích thước dữ liệu
In ra số dòng, số cột và thông tin bộ nhớ sử dụng của hai bảng chính.

In [ ]:
for name, df in [('orders', df_orders), ('order_items', df_order_items)]:
    mem = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f'Bảng {name}:')
    print(f'  - Số dòng: {df.shape[0]:,}')
    print(f'  - Số cột: {df.shape[1]}')
    print(f'  - Bộ nhớ sử dụng: {mem:.2f} MB')


### 3. Khảo sát cấu trúc
Khảo sát danh sách cột, kiểu dữ liệu, thống kê loại kiểu dữ liệu và tỷ lệ giá trị khác nhau (Unique Ratio = nunique / total rows).

In [ ]:
for name, df in [('orders', df_orders), ('order_items', df_order_items)]:
    print(f'\n=================== CẤU TRÚC BẢNG {name.upper()} ===================')
    print('Danh sách cột:', list(df.columns))
    print('\nThống kê kiểu dữ liệu:')
    print(df.dtypes.value_counts())
    
    unique_stats = []
    for col in df.columns:
        nunique = df[col].nunique()
        ratio = nunique / len(df) * 100
        unique_stats.append({
            'Cột': col,
            'Kiểu dữ liệu': str(df[col].dtype),
            'Nunique': nunique,
            'Unique Ratio (%)': f'{ratio:.4f}%'
        })
    display(pd.DataFrame(unique_stats))


In [3]:
# Code sử dụng .info() cho các bảng
for name, df in dataframes.items():
    print(f'\n--- Thông tin bảng {name.upper()} ---')
    df.info()



--- Thông tin bảng CUSTOMERS ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

--- Thông tin bảng GEOLOCATION ---
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocatio

**Nhận xét:**
- **Kiểu dữ liệu:** Các cột ngày tháng/thời gian (như `order_purchase_timestamp`, `order_approved_at`, `order_delivered_customer_date` trong bảng `orders`, `shipping_limit_date` trong `order_items`, `review_creation_date` & `review_answer_timestamp` trong `order_reviews`) hiện tại đang ở kiểu `object` (string). Cần chuyển đổi sang kiểu `datetime` ở bước tiền xử lý để phân tích chuỗi thời gian chính xác.
- **Giá trị khuyết thiếu (Null):**
  - Bảng `order_reviews`: Cột `review_comment_title` (87,656 dòng null) và `review_comment_message` (58,247 dòng null) bị thiếu rất nhiều. Đây là hiện tượng bình thường vì phần lớn khách hàng chỉ đánh giá điểm số (score) mà không viết tiêu đề hoặc nội dung đánh giá chi tiết.
  - Bảng `orders`: Có một số lượng nhỏ giá trị khuyết thiếu ở các cột `order_approved_at` (160 nulls), `order_delivered_carrier_date` (1,783 nulls), và `order_delivered_customer_date` (2,965 nulls). Những dòng này có thể phản ánh các đơn hàng bị hủy, đang xử lý hoặc chưa được giao thành công.
  - Bảng `products`: Có 610 giá trị null trong các cột tên danh mục (`product_category_name`), độ dài tên/mô tả và số lượng ảnh. Ngoài ra có 2 giá trị khuyết ở các cột kích thước/trọng lượng.


### 4. Thống kê mô tả dữ liệu
Sử dụng describe() để phân tích các đặc trưng số và danh mục trong bảng chính.

In [5]:
# Code mô tả thống kê
for name, df in dataframes.items():
    print(f'\n--- Thống kê mô tả bảng {name.upper()} ---')
    display(df.describe())



--- Thống kê mô tả bảng CUSTOMERS ---


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000



--- Thống kê mô tả bảng GEOLOCATION ---


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06,1.000163e+06
mean,3.657417e+04,-2.117615e+01,-4.639054e+01
std,3.054934e+04,5.715866e+00,4.269748e+00
min,1.001000e+03,-3.660537e+01,-1.014668e+02
25%,1.107500e+04,-2.360355e+01,-4.857317e+01
50%,2.653000e+04,-2.291938e+01,-4.663788e+01
75%,6.350400e+04,-1.997962e+01,-4.376771e+01
max,9.999000e+04,4.506593e+01,1.211054e+02



--- Thống kê mô tả bảng ORDER_ITEMS ---


...

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000



--- Thống kê mô tả bảng SELLERS ---


,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000



--- Thống kê mô tả bảng CATEGORY_TRANSLATION ---


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


**Nhận xét:**
- **Bảng `order_items`:**
  - Cột `price` (giá sản phẩm) dao động từ 0.85 đến 6,735.00, với độ lệch chuẩn lớn (~183.63) so với mức trung bình (~120.65). Mức giá tối đa 6,735.00 là một giá trị ngoại lai (outlier) rất lớn cần lưu ý.
  - Cột `freight_value` (phí vận chuyển) có giá trị nhỏ nhất là 0.00 (miễn phí vận chuyển) và lớn nhất lên tới 409.68 (cao hơn nhiều so với giá trị trung bình 19.99). Điều này có thể do các mặt hàng cồng kềnh hoặc khoảng cách giao hàng xa.
- **Bảng `order_payments`:**
  - Cột `payment_installments` (số kỳ trả góp) có giá trị lớn nhất là 24 tháng. Giá trị nhỏ nhất bằng 0 là một điểm bất thường cần làm rõ (thường trả góp tối thiểu phải từ 1 kỳ trở lên).
  - Cột `payment_value` (giá trị thanh toán) có mức tối đa là 13,664.08, lớn hơn rất nhiều so với mức trung bình (~154.10).
- **Bảng `order_reviews`:**
  - Cột `review_score` có điểm trung bình là 4.09, trung vị (50%) và 75% đều đạt điểm tối đa là 5.0. Điều này cho thấy mức độ hài lòng chung của khách hàng ở mức cao, tuy nhiên vẫn có điểm tối thiểu là 1.0.


## IV. Kiểm tra chất lượng dữ liệu của bảng chính
### 1. Missing Values & 2. Duplicate & 3. Giá trị sai logic (bất thường)
Kiểm tra chất lượng dữ liệu thô, tìm kiếm các giá trị khuyết, trùng lặp và các lỗi logic nghiệp vụ.

In [ ]:
for name, df in [('orders', df_orders), ('order_items', df_order_items)]:
    print(f'\n=================== CHẤT LƯỢNG BẢNG {name.upper()} ===================')
    # 1. Missing Values
    null_counts = df.isnull().sum()
    null_percent = (null_counts / len(df)) * 100
    missing_df = pd.DataFrame({
        'Số lượng Missing': null_counts,
        'Tỷ lệ Missing (%)': null_percent.round(4)
    })
    print('Thông tin Missing Values:')
    display(missing_df[missing_df['Số lượng Missing'] > 0])
    
    # 2. Duplicate
    dup = df.duplicated().sum()
    print(f'Số lượng dòng trùng lặp: {dup}')
    
# 3. Giá trị sai logic (bất thường)
print('\n--- KIỂM TRA GIÁ TRỊ BẤT THƯỜNG ---')
invalid_price = (df_order_items['price'] < 0).sum()
invalid_freight = (df_order_items['freight_value'] < 0).sum()
print(f'Số lượng sản phẩm có price âm: {invalid_price}')
print(f'Số lượng sản phẩm có freight_value âm: {invalid_freight}')

# Đơn hàng giao trước khi đặt mua
df_orders_temp = df_orders.copy()
df_orders_temp['order_purchase_timestamp'] = pd.to_datetime(df_orders_temp['order_purchase_timestamp'])
df_orders_temp['order_delivered_customer_date'] = pd.to_datetime(df_orders_temp['order_delivered_customer_date'])
invalid_delivery = (df_orders_temp['order_delivered_customer_date'] < df_orders_temp['order_purchase_timestamp']).sum()
print(f'Số lượng đơn hàng giao trước khi mua: {invalid_delivery}')


In [4]:
# Code kiểm tra trùng lặp
print('Số lượng dòng trùng lặp trong các bảng:')
for name, df in dataframes.items():
    duplicates = df.duplicated().sum()
    print(f'- {name}: {duplicates} dòng trùng')


Số lượng dòng trùng lặp trong các bảng:
- customers: 0 dòng trùng
- geolocation: 261831 dòng trùng
- order_items: 0 dòng trùng
- order_payments: 0 dòng trùng
- order_reviews: 0 dòng trùng
- orders: 0 dòng trùng
- products: 0 dòng trùng
- sellers: 0 dòng trùng
- category_translation: 0 dòng trùng


**Nhận xét:**
- Hầu hết các bảng chính như `customers`, `orders`, `order_items`, `order_payments`, `order_reviews`, `products`, `sellers`, `category_translation` đều có **0 dòng trùng lặp**, cho thấy dữ liệu giao dịch và thông tin định danh rất sạch sẽ.
- Riêng bảng `geolocation` có **261,831 dòng trùng lặp** (~26.2%). Điều này là hợp lý vì các tọa độ địa lý có thể được ghi nhận nhiều lần cho cùng một tiền tố mã bưu chính (zip code prefix), thành phố và bang. Tuy nhiên, khi kết nối dữ liệu địa lý, chúng ta cần nhóm (aggregate) hoặc loại bỏ trùng lặp để tránh làm tăng số lượng dòng ngoài ý muốn.


## V. Khảo sát các nhóm dữ liệu của bảng chính
Khảo sát các nhóm dữ liệu chính: Numeric, Category, Date, và Target variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric columns
print('--- Nhóm Numeric (price, freight_value) ---')
display(df_order_items[['price', 'freight_value']].describe())

# Category columns
print('\n--- Nhóm Category (order_status) ---')
print(df_orders['order_status'].value_counts())

# Date columns
print('\n--- Nhóm Date (order_purchase_timestamp) ---')
min_date = df_orders_temp['order_purchase_timestamp'].min()
max_date = df_orders_temp['order_purchase_timestamp'].max()
print(f'Ngày đặt hàng nhỏ nhất: {min_date}')
print(f'Ngày đặt hàng lớn nhất: {max_date}')
print(f'Khoảng thời gian: {max_date - min_date}')

# Target column Analysis
print('\n--- Target Variable (delivery_time) ---')
delivery_days = (df_orders_temp['order_delivered_customer_date'] - df_orders_temp['order_purchase_timestamp']).dt.days
print('Mô tả thống kê của Target (ngày):')
print(delivery_days.describe())
print(f'Số lượng Target bị khuyết (chưa giao hàng): {delivery_days.isnull().sum()} ({delivery_days.isnull().mean()*100:.2f}%)')

# Vẽ histogram target
plt.figure(figsize=(10, 4))
sns.histplot(delivery_days.dropna(), bins=50, kde=True, color='purple')
plt.title('Phân phối thời gian giao hàng thực tế (Target)')
plt.xlabel('Số ngày giao hàng')
plt.ylabel('Tần suất')
plt.xlim(0, 60)
plt.show()


## VI. Khảo sát dữ liệu của bảng phụ
Khảo sát các thông tin cấu trúc, kiểu dữ liệu, thống kê mô tả, missing values, duplicates, và bộ nhớ của 7 bảng phụ.

In [ ]:
for name, df in dataframes.items():
    if name in ['orders', 'order_items']:
        continue
    print(f'\n=================== KHẢO SÁT BẢNG PHỤ: {name.upper()} ===================')
    print(f'Kích thước: {df.shape}')
    print('Danh sách cột:', list(df.columns))
    print('Bộ nhớ sử dụng:', f'{df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')
    print('Số lượng dòng trùng lặp:', df.duplicated().sum())
    print('Thông tin missing values:')
    null_df = df.isnull().sum()
    display(null_df[null_df > 0])
    
    # Categorical column analysis if any object/category columns exist
    cat_cols = df.select_dtypes(include=['object']).columns
    if len(cat_cols) > 0:
        print('Thông tin cột phân loại (nunique):')
        for col in cat_cols[:3]: # limit to top 3 for clean view
            print(f'  - {col}: {df[col].nunique()} giá trị duy nhất')


## VII. Kiểm tra quan hệ giữa các bảng
Kiểm tra tính duy nhất của khóa chính (Primary Key), tính toàn vẹn của khóa ngoại (Foreign Key), và kiểm thử Join.

In [6]:
# Code kiểm tra tính duy nhất (is_unique)
print('Kiểm tra tính duy nhất của một số khóa chính giả định:')
print('- customers (customer_id):', df_customers['customer_id'].is_unique)
print('- orders (order_id):', df_orders['order_id'].is_unique)
print('- products (product_id):', df_products['product_id'].is_unique)
print('- sellers (seller_id):', df_sellers['seller_id'].is_unique)


Kiểm tra tính duy nhất của một số khóa chính giả định:
- customers (customer_id): True
- orders (order_id): True
- products (product_id): True
- sellers (seller_id): True


**Nhận xét:**
- Các trường khóa chính giả định bao gồm: `customer_id` (bảng `customers`), `order_id` (bảng `orders`), `product_id` (bảng `products`), và `seller_id` (bảng `sellers`) đều trả về kết quả **True** cho thuộc tính `.is_unique`.
- Điều này xác nhận các cột này hoàn toàn duy nhất, đảm bảo tính toàn vẹn thực thể và có thể dùng làm khóa chính đáng tin cậy để liên kết các bảng với nhau trong mô hình dữ liệu.


In [ ]:
# Foreign Key Integrity Check
print('--- KIỂM TRA KHÓA NGOẠI (FOREIGN KEY) ---')
missing_orders = ~df_order_items['order_id'].isin(df_orders['order_id'])
print(f'Số khóa ngoại order_id bị thiếu ở order_items: {missing_orders.sum()}')

missing_products = ~df_order_items['product_id'].isin(df_products['product_id'])
print(f'Số khóa ngoại product_id bị thiếu ở order_items: {missing_products.sum()}')

missing_sellers = ~df_order_items['seller_id'].isin(df_sellers['seller_id'])
print(f'Số khóa ngoại seller_id bị thiếu ở order_items: {missing_sellers.sum()}')

# Join Validation
print('\n--- KIỂM TRA LIÊN KẾT JOIN ---')
merged_df = pd.merge(df_orders, df_order_items, on='order_id', how='inner')
print(f'Số dòng sau INNER JOIN (orders + order_items): {len(merged_df):,}')
print(f'Số dòng ban đầu của order_items: {len(df_order_items):,}')
print(f'Mất mát dữ liệu sau Join: {len(df_order_items) - len(merged_df)} dòng')


## VIII. Đánh giá sơ bộ
- **Những cột được giả định là quan trọng:**
  - `order_purchase_timestamp` và `order_delivered_customer_date` (để tính toán biến mục tiêu thời gian giao hàng).
  - `customer_zip_code_prefix` và `seller_zip_code_prefix` (để liên kết geolocation tính toán khoảng cách địa lý).
  - `price`, `freight_value` và `payment_value` (các thuộc tính tài chính liên quan đến chi phí).
- **Những cột có nhiều Missing:**
  - `review_comment_title` (88.34%) và `review_comment_message` (58.70%) trong bảng `order_reviews`.
- **Những cột cần xử lý:**
  - Định dạng lại các cột ngày tháng/thời gian về kiểu `datetime`.
  - Điền khuyết thiếu cho các cột thời gian giao hàng và các thông tin sản phẩm bị null.
- **Những cột cần Feature Engineering:**
  - Tính khoảng cách địa lý Haversine dựa trên geolocation của khách hàng và người bán.
  - Tách đặc trưng thời gian (giờ đặt hàng, ngày đặt hàng, tháng đặt hàng, thứ trong tuần).
  - Phân cụm vị trí địa lý của khách hàng để tối ưu hóa vị trí đặt kho bãi.

*Lưu ý: Đây chỉ là đánh giá ban đầu, chưa phải kết luận cuối cùng. Các giả thuyết sẽ được kiểm chứng trong Notebook 04 (EDA) và Notebook 05 (Feature Engineering).*


## IX. Kết luận
- **Quy mô dữ liệu:** Dataset gồm 9 bảng dữ liệu liên kết chéo với nhau. Bảng chính `orders` có 99,441 bản ghi, bảng `order_items` có 112,650 bản ghi.
- **Tình trạng Missing:** Không có missing ở các cột ID khóa chính. Missing chủ yếu xuất hiện ở các trường bình luận đánh giá không bắt buộc và mốc thời gian giao hàng của đơn chưa hoàn thành.
- **Tình trạng Duplicate:** Không có duplicate dòng ở các bảng giao dịch chính. Chỉ xuất hiện duplicate ở bảng geolocation thô (mã bưu chính lặp lại) và đã được ghi nhận để xử lý.
- **Sẵn sàng cho bước tiếp theo:** Dữ liệu thô hoàn toàn sạch sẽ và sẵn sàng đưa vào cơ sở dữ liệu PostgreSQL.
- **Chuẩn bị cho Notebook 02:** Thiết lập đường ống dữ liệu (ETL pipeline) để import toàn bộ 9 bảng dữ liệu vào PostgreSQL và thực hiện các phép JOIN hiệu năng cao.


### Trường hợp Dataset có quá nhiều cột (Ví dụ từ 150-300 cột)
- Với các bài toán có quy mô số lượng cột cực kỳ lớn, nhóm sẽ áp dụng các quy tắc tối ưu sau để khảo sát:
  1. **Chia nhóm cột:** Gom nhóm theo nghiệp vụ (thông tin khách hàng, thông tin tài chính, lịch sử tín dụng, thời gian, v.v.).
  2. **Chỉ phân tích cột quan trọng:** Tập trung vào các cột ID, Target, Date, Foreign Key và các cột mang ý nghĩa nghiệp vụ chính.
  3. **Thống kê theo nhóm cột:** In số lượng các cột kiểu Numeric, Category, Date, ID thay vì in chi tiết từng cột.
  4. **Không in toàn bộ dữ liệu:** Không chạy `.describe()` hoặc `.value_counts()` bừa bãi cho hàng trăm cột để tránh làm quá tải giao diện.


## Pipeline của dự án
```
Notebook 01: Data Understanding
        ↓
Notebook 02: PostgreSQL & Data Pipeline
        ↓
Notebook 03: Data Cleaning
        ↓
Notebook 04: EDA & Visualization
        ↓
Notebook 05: Feature Engineering
        ↓
Notebook 06: Machine Learning
        ↓
Notebook 07: AI Deployment (FastAPI + Streamlit)
```


## Trả lời 4 câu hỏi cốt lõi
1. **Dữ liệu đang có là gì?** Dữ liệu thương mại điện tử Olist Brazil từ 2016-2018 gồm thông tin khách hàng, đơn đặt hàng, chi tiết thanh toán, đánh giá, chi tiết mặt hàng, sản phẩm, người bán và tọa độ địa lý.
2. **Dữ liệu có đáng tin cậy không?** Rất đáng tin cậy. Các bảng chính không có dòng trùng lặp, các khóa chính là duy nhất và nhất quán. Tỷ lệ missing phù hợp với thực tế vận hành.
3. **Dữ liệu được tổ chức và liên kết như thế nào?** Tổ chức dưới dạng lược đồ quan hệ hình sao/bông tuyết, liên kết chặt chẽ thông qua các cột ID khóa chính và khóa ngoại.
4. **Cần chuẩn bị gì cho các bước tiếp theo?** Đưa dữ liệu thô vào PostgreSQL để lưu trữ tập trung và tối ưu hóa các thao tác xử lý liên kết dữ liệu quy mô lớn (JOIN).
